In [ ]:
# ---- COMPLETE STANDALONE PART V ----
!pip install ultralytics -q

from google.colab import drive
drive.mount('/content/drive')

import os, time, yaml
from ultralytics import YOLO
import pandas as pd

DRIVE = '/content/drive/MyDrive'
RESULTS_DIR = f'{DRIVE}/yolo_results'
VISDRONE_YOLO = f'{DRIVE}/visdrone_yolo'
yaml_path = f'{VISDRONE_YOLO}/visdrone.yaml'
CLASS_NAMES = ['pedestrian','people','bicycle','car','van','truck',
               'tricycle','awning-tricycle','bus','motor']

# Recreate yaml
os.makedirs(VISDRONE_YOLO, exist_ok=True)
with open(yaml_path, 'w') as f:
    yaml.dump({'path': VISDRONE_YOLO, 'train': 'images/train',
               'val': 'images/val', 'nc': 10, 'names': CLASS_NAMES}, f)
print(f'YAML ready: {yaml_path}')

def train_yolo(model_name, run_name, epochs=50, imgsz=640, batch=16, lr=0.01):
    model = YOLO(model_name)
    start = time.time()
    model.train(data=yaml_path, epochs=epochs, imgsz=imgsz, batch=batch,
                lr0=lr, optimizer='SGD', project=RESULTS_DIR, name=run_name,
                exist_ok=True, patience=15, save=True, plots=True, verbose=False)
    elapsed = time.time() - start
    val = model.val(data=yaml_path, split='val', verbose=False)
    metrics = {
        'model': run_name,
        'mAP50':          round(float(val.box.map50), 4),
        'mAP50-95':       round(float(val.box.map),   4),
        'precision':      round(float(val.box.mp),    4),
        'recall':         round(float(val.box.mr),    4),
        'train_time_hrs': round(elapsed / 3600, 2)
    }
    print(f"\nResults for {run_name}:")
    for k, v in metrics.items():
        print(f"  {k}: {v}")
    return metrics

# Hardcoded baseline from previous run
baseline = {'model': 'YOLOv11n', 'mAP50': 0.3027, 'mAP50-95': 0.1702,
            'precision': 0.4376, 'recall': 0.3260, 'train_time_hrs': 0.77}

print('Training YOLOv8n...')
v8  = train_yolo('yolov8n.pt',  'compare_yolov8n')
print('Training YOLOv9c...')
v9  = train_yolo('yolov9c.pt',  'compare_yolov9c')
print('Training YOLOv10n...')
v10 = train_yolo('yolov10n.pt', 'compare_yolov10n')
print('Training YOLOv5nu...')
v5  = train_yolo('yolov5nu.pt', 'compare_yolov5nu')

# Save final table
df = pd.DataFrame([
    {**baseline,  'model': 'YOLOv11n'},
    {**v8,        'model': 'YOLOv8n'},
    {**v9,        'model': 'YOLOv9c'},
    {**v10,       'model': 'YOLOv10n'},
    {**v5,        'model': 'YOLOv5nu'},
])
print('\n===== PART V — Final Comparison =====')
print(df.to_string(index=False))
df.to_csv(f'{RESULTS_DIR}/part5_comparison.csv', index=False)
print('Saved to Drive!')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 61.0 MB/s eta 0:00:00
Mounted at /content/drive
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
YAML ready: /content/drive/MyDrive/visdrone_yolo/visdrone.yaml
Training YOLOv8n...
Ultralytics 8.4.37 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/visdrone_yolo/visdrone.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=